**Note**: 

> This exercise has been written out in something called a Jupyter Notebook. We'll discuss Jupyter Notebooks in more detail later in this specialization—they are very a powerful tool for data science communication!—but for the time being, the notebook is just a convenient way for us to write out the exercise. You don't need to *do* anything with the notebook except read its contents—just use write your Python code in a regular `.py` file.

# Exercise: queries into the state of electricity production and CO2 emissions in the United States

In data science, we often need to have a sense of the idiosyncrasies of the data, how they relate to the questions we are trying to answer, and to use that information to help us to determine what approach, such as machine learning, we may need to apply to achieve our goal. This exercise provides practice in exploring a dataset and answering questions that might arise from applications related to the data.

**Data**. The data for this problem can be found in the `data` folder. The filename is `egrid2016.csv`. This dataset is the U.S. Environmental Protection Agency's (EPA) [Emissions & Generation Resource Integrated Database (eGRID)](https://www.epa.gov/energy/emissions-generation-resource-integrated-database-egrid) containing information about all power plants in the United States, the amount of electricity they generate, what fuel they use, emissions produced, the location of the plant, and many more quantities. We'll be using a subset of those data.

The fields we'll be using include:					
    
|field    |description|
|:-----   |:-----|
|SEQPLT16 |eGRID2016 Plant file sequence number (the index)| 
|PSTATABB |Plant state abbreviation|
|PNAME    |Plant name |
|LAT      |Plant latitude |
|LON      |Plant longitude|
|PLPRMFL  |Plant primary fuel |
|NAMEPCAP |Plant nameplate capacity (Megawatts MW)|
|PLNGENAN |Plant annual net generation (Megawatt-hours MWh)|
|PLCO2EQA |Plant annual CO2 equivalent emissions (tons)|

For more details on the data, you can refer to the [eGrid technical documents](https://www.epa.gov/sites/default/files/2021-02/documents/egrid2019_technical_guide.pdf). For example, you may want to review page 45 and the section "Plant Primary Fuel (PLPRMFL)", which gives the full names of the fuel types including WND for wind, NG for natural gas, BIT for Bituminous coal, etc. Codebooks for data are common in data science when the variable names would otherwise be onerously long.

In [2]:
import pandas as pd
pd.set_option("mode.copy_on_write", True)
df=pd.read_csv("egrid2016.csv")
df.head()

,SEQPLT16,PSTATABB,PNAME,LAT,LON,PLPRMFL,NAMEPCAP,PLNGENAN,PLCO2EQA
0,1,AK,7-Mile Ridge Wind Project,63.210689,-143.247156,WND,1.8,0.0,0.00
1,2,AK,Agrium Kenai Nitrogen Operations,60.673200,-151.378400,NG,21.6,0.0,0.00
2,3,AK,Alakanuk,62.683300,-164.654400,DFO,2.6,1213.0,1049.86
3,4,AK,Allison Creek Hydro,61.084444,-146.353333,WAT,6.5,881.0,0.00
4,5,AK,Ambler,67.087980,-157.856719,DFO,1.1,1316.0,1087.88



**Your objective**. For this dataset, your goal is to answer the following questions about electricity generation in the United States by constructing appropriate queries of the data:

1. Which power plant generated the most energy in 2016 (measured in MWh)? Since there is a column with the annual generation ('PLNGENAN'), consider how this can be used to answer this question.

2. Which power plant produced the most CO2 emissions (measured in tons)? 

3. In what state is the plant with the most CO2 emissions (question 2) located?

4. What is the primary fuel of the plant with the most CO2 emissions?

5. What is the name of the northern-most power plant in the United States? (hint: latitude is the quantity that measures how far north or south a location is across the globe)

6. In what state is the northern-most power plant in the United States located?

7. Which state has the largest number of hydroelectric plants? In this case, each power plant counts once so regardless of how large the power plant is, we want to determine which state has the most of them. Note the primary fuel for hydroelectric plants is listed as water in the documentation.

8. How many hydroelectric plants does the state with the most (which you identified in the last question) have?

9. Which state(s) has generated the most *energy* (MWh) using coal? If there are more than one, list the state abbreviations in alphabetical order, separated with commas (but no spaces). You may also want to explore the documentation for the `isin()` method for `pandas`. Note: in the eGrid documentation, there are multiple types of coal listed; be sure to factor in each type of coal. 

10. How much energy (in MWh) do the plants in question 9 produce in total? Please round to the nearest whole number.

11. Which states have EXACTLY 1 coal plant? List the state abbreviations in alphabetical order, separated with commas (but no spaces).

12. Which primary fuel produced the *most* CO2 emissions in the United States? We would like to compare natural gas, coal, oil, and renewables but the current categories are much more specific than that. As a first step, group the data as shown below, replacing the existing labels with the replacements suggested. For example, BIT and LIG should be replaced with COAL.
- COAL = BIT, LIG, RC, SUB, WC
- OIL = DFO, JF, KER, RFO, WO
- GAS = BFG, COG, LFG, NG, OG, PG, PRG 
- RENEW = GEO, SUN, WAT, WDL, WDS, WND

You may want to create a function that does this replacement prior to running your code. You can check whether or not it was successful by verifying that each of the values that should be replaced has been replaced - check that before moving on with the question.

You will want to use 'PLCO2EQA' to answer this question as it's the quantity of emissions each plant generates.

> **Note your responses to each of these questions. You will be asked about these on the final quiz this week.**

In [3]:
df=df.rename(columns={
    'PLPRMFL':'Primary_Fuel',
    'NAMEPCAP':'Nameplate_capacity_MW',
    'PLNGENAN':'Annual_generation_MWH',
    'PLCO2EQA':'Annual_CO2_tons',
    'PSTATABB':'State'
})
df.head()    

,SEQPLT16,State,PNAME,LAT,LON,Primary_Fuel,Nameplate_capacity_MW,Annual_generation_MWH,Annual_CO2_tons
0,1,AK,7-Mile Ridge Wind Project,63.210689,-143.247156,WND,1.8,0.0,0.00
1,2,AK,Agrium Kenai Nitrogen Operations,60.673200,-151.378400,NG,21.6,0.0,0.00
2,3,AK,Alakanuk,62.683300,-164.654400,DFO,2.6,1213.0,1049.86
3,4,AK,Allison Creek Hydro,61.084444,-146.353333,WAT,6.5,881.0,0.00
4,5,AK,Ambler,67.087980,-157.856719,DFO,1.1,1316.0,1087.88


In [4]:
#1 Which power plant generated the most energy in 2016 (measured in MWh)? Since there is a column with the annual generation ('PLNGENAN'), 
#consider how this can be used to answer this question.
a=df.Annual_generation_MWH.max()
df.loc[df['Annual_generation_MWH']==a]

,SEQPLT16,State,PNAME,LAT,LON,Primary_Fuel,Nameplate_capacity_MW,Annual_generation_MWH,Annual_CO2_tons
390,391,AZ,Palo Verde,33.3881,-112.8617,NUC,4209.6,32377477.0,0.0


In [5]:
#2. Which power plant produced the most CO2 emissions (measured in tons)? 

b=df.Annual_CO2_tons.max()
df.loc[df['Annual_CO2_tons']==b]
#p=df.loc[df.Annual_CO2_tons=='WAT']
#p=p[['PSTATABB','PLCO2EQA']]
#p.value_counts()


,SEQPLT16,State,PNAME,LAT,LON,Primary_Fuel,Nameplate_capacity_MW,Annual_generation_MWH,Annual_CO2_tons
191,192,AL,James H Miller Jr,33.6319,-87.0597,SUB,2822.0,18193039.0,21724990.49


In [6]:
#3. In what state is the plant with the most CO2 emissions (question 2) located?

#AL

In [7]:
#4. What is the primary fuel of the plant with the most CO2 emissions?

#SUB

In [8]:
#5. What is the name of the northern-most power plant in the United States? (hint: latitude is the quantity that measures how far north or 
#south a location is across the globe)

c=df.LAT.max()
df.loc[df['LAT']==c]

,SEQPLT16,State,PNAME,LAT,LON,Primary_Fuel,Nameplate_capacity_MW,Annual_generation_MWH,Annual_CO2_tons
11,12,AK,Barrow,71.292,-156.7786,NG,20.3,50162.0,44205.17


In [9]:
#6 In what state is the northern-most power plant in the United States located?

#AK

In [23]:
#7. Which state has the largest number of hydroelectric plants? In this case, each power plant counts once so regardless
#of how large the power plant is, we want to determine which state has the most of them. Note the primary fuel for 
#hydroelectric plants is listed as water in the documentation.

d=df[df['Primary_Fuel']=='WAT']
d.State.value_counts()

#CA

State
CA    264
NY    165
WA     80
ID     76
OR     69
WI     63
ME     58
MI     56
CO     49
VT     47
NC     41
SC     35
AK     35
NH     34
GA     33
MA     31
UT     30
TN     29
MT     26
TX     25
VA     25
AL     24
MN     23
PA     20
AR     19
WY     16
CT     15
AZ     12
WV     12
NE     11
OK     11
IL     10
KY     10
MO      8
HI      8
NV      6
OH      5
IN      5
SD      4
NM      4
IA      4
NJ      3
RI      2
FL      2
MD      2
KS      1
ND      1
LA      1
Name: count, dtype: int64

In [ ]:
#8 How many hydroelectric plants does the state with the most (which you identified in the last question) have?

In [24]:
#264

In [25]:
#9. Which state(s) has generated the most *energy* (MWh) using coal? If there are more than one, list the state abbreviations 
#in alphabetical order, separated with commas (but no spaces). You may also want to explore the documentation for the `isin()` 
#method for `pandas`. Note: in the eGrid documentation, there are multiple types of coal listed; be sure to factor in each type of coal.

df.head()

,SEQPLT16,State,PNAME,LAT,LON,Primary_Fuel,Nameplate_capacity_MW,Annual_generation_MWH,Annual_CO2_tons
0,1,AK,7-Mile Ridge Wind Project,63.210689,-143.247156,WND,1.8,0.0,0.00
1,2,AK,Agrium Kenai Nitrogen Operations,60.673200,-151.378400,NG,21.6,0.0,0.00
2,3,AK,Alakanuk,62.683300,-164.654400,DFO,2.6,1213.0,1049.86
3,4,AK,Allison Creek Hydro,61.084444,-146.353333,WAT,6.5,881.0,0.00
4,5,AK,Ambler,67.087980,-157.856719,DFO,1.1,1316.0,1087.88


In [49]:
#df1=df.groupby(['State','Primary_Fuel'])['Annual_generation_MWH'].sum()
#df1
df.query("Primary_Fuel"==["BIT","LIG","RC","SGC","SUB","WC"])

ValueError: expr must be a string to be evaluated, <class 'bool'> given

In [104]:
#From Step 9: Which state(s) has generated the most energy  (MWh) using coal? 

#m=['BIT','LIG','RC','SGC','SUB','WC']
df['coal']=df[(df['PLPRMFL']=='BIT')+(df['PLPRMFL']=='LIG')+(df['PLPRMFL']=='RC')+(df['PLPRMFL']=='SGC')+(df['PLPRMFL']=='SUB')+(df['PLPRMFL']=='WC')]['PLCO2EQA'].sum()
df.head()
#p=df[['PSTATABB','PLCO2EQA','PLPRMFL','coal']]
#p.to_csv("out3.csv")

#d = p.pivot_table(
#    values="PLCO2EQA", index="PSTATABB", columns="coal", aggfunc="sum"
#)
#d.head()

#p=df.loc[df.PLPRMFL.isin(m)]

#p.value_counts()

,SEQPLT16,PSTATABB,PNAME,LAT,LON,PLPRMFL,NAMEPCAP,PLNGENAN,PLCO2EQA,coal
0,1,AK,7-Mile Ridge Wind Project,63.210689,-143.247156,WND,1.8,0.0,0.00,1.401523e+09
1,2,AK,Agrium Kenai Nitrogen Operations,60.673200,-151.378400,NG,21.6,0.0,0.00,1.401523e+09
2,3,AK,Alakanuk,62.683300,-164.654400,DFO,2.6,1213.0,1049.86,1.401523e+09
3,4,AK,Allison Creek Hydro,61.084444,-146.353333,WAT,6.5,881.0,0.00,1.401523e+09
4,5,AK,Ambler,67.087980,-157.856719,DFO,1.1,1316.0,1087.88,1.401523e+09


In [133]:
df1=df.loc[df['PLPRMFL'].isin(['BIT','LIG','RC','SGC','SUB','WC'])]
#df1['PLNGENAN'].sum()
df2=df1[['PSTATABB','PLNGENAN']]
print(df2.value_counts())

PSTATABB  PLNGENAN 
OH        0.0          7
WV        0.0          7
PA        0.0          6
KY        0.0          4
WI        0.0          4
                      ..
KY        6617798.0    1
          3667297.0    1
          3114894.0    1
          2736301.0    1
MA        0.0          1
Name: count, Length: 404, dtype: int64
